# Convert Snowflake Arctic Embed XS → float32 TFLite
Uses TensorFlow directly (already in Colab — no problematic installs).
Run cells top-to-bottom. GPU not required.

In [ ]:
# Cell 1 — Install (minimal — just pin transformers; TF is already in Colab)
!pip install -q "transformers>=4.40" sentencepiece

In [ ]:
# Cell 2 — Load model in TensorFlow (converts PyTorch weights automatically)
import tensorflow as tf
from transformers import TFAutoModel, AutoTokenizer

MODEL_ID = "Snowflake/snowflake-arctic-embed-xs"
SEQ_LEN  = 128

print(f"TensorFlow {tf.__version__}")
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

print("Loading model (converts PyTorch weights to TF — ~1 min)...")
tf_model = TFAutoModel.from_pretrained(MODEL_ID, from_pt=True)
print("Model loaded.")

In [ ]:
# Cell 3 — Wrap with a fixed-shape signature and save as SavedModel
import os

class EmbedWrapper(tf.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    @tf.function(input_signature=[
        tf.TensorSpec(shape=[1, SEQ_LEN], dtype=tf.int32, name="input_ids"),
        tf.TensorSpec(shape=[1, SEQ_LEN], dtype=tf.int32, name="attention_mask"),
        tf.TensorSpec(shape=[1, SEQ_LEN], dtype=tf.int32, name="token_type_ids"),
    ])
    def __call__(self, input_ids, attention_mask, token_type_ids):
        out = self.model(
            input_ids=input_ids,
            attention_mask=attention_mask,
            token_type_ids=token_type_ids,
            training=False,
        )
        return {"last_hidden_state": out.last_hidden_state}

wrapper = EmbedWrapper(tf_model)

SAVED_MODEL_DIR = "/content/arctic_saved_model"
print("Saving SavedModel...")
tf.saved_model.save(
    wrapper,
    SAVED_MODEL_DIR,
    signatures={"serving_default": wrapper.__call__},
)
print(f"Saved to {SAVED_MODEL_DIR}")

In [ ]:
# Cell 4 — Convert SavedModel → TFLite float32
converter = tf.lite.TFLiteConverter.from_saved_model(
    SAVED_MODEL_DIR,
    signature_keys=["serving_default"],
)
converter.target_spec.supported_ops = [
    tf.lite.OpsSet.TFLITE_BUILTINS,
    tf.lite.OpsSet.SELECT_TF_OPS,   # needed for some BERT ops (LayerNorm etc.)
]
converter._experimental_lower_tensor_list_ops = False

print("Converting (may take 2-3 min)...")
tflite_bytes = converter.convert()

TFLITE_PATH = "/content/snowflake-arctic-embed-xs.tflite"
with open(TFLITE_PATH, "wb") as f:
    f.write(tflite_bytes)

size_mb = os.path.getsize(TFLITE_PATH) / 1024 / 1024
print(f"Done! {TFLITE_PATH}  ({size_mb:.1f} MB)")

In [ ]:
# Cell 5 — Inspect tensor names (paste output into chat so Kotlin code can be updated)
interp = tf.lite.Interpreter(model_path=TFLITE_PATH)
interp.allocate_tensors()

print("=== INPUT TENSORS ===")
for t in interp.get_input_details():
    print(f"  [{t['index']}]  name='{t['name']}'  shape={t['shape']}  dtype={t['dtype']}")

print("\n=== OUTPUT TENSORS ===")
for t in interp.get_output_details():
    print(f"  [{t['index']}]  name='{t['name']}'  shape={t['shape']}  dtype={t['dtype']}")

In [ ]:
# Cell 6 — Sanity-check: semantic similarity test
import numpy as np

def embed(text: str) -> np.ndarray:
    enc = tokenizer(text, return_tensors="np", max_length=SEQ_LEN,
                    padding="max_length", truncation=True)
    for t in interp.get_input_details():
        name = t["name"]
        if   "input_ids"      in name: interp.set_tensor(t["index"], enc["input_ids"].astype(np.int32))
        elif "attention_mask" in name: interp.set_tensor(t["index"], enc["attention_mask"].astype(np.int32))
        elif "token_type_ids" in name: interp.set_tensor(t["index"], enc["token_type_ids"].astype(np.int32))
    interp.invoke()
    last_hidden = interp.get_tensor(interp.get_output_details()[0]["index"])  # [1, 128, 384]
    mask = enc["attention_mask"][0].astype(np.float32)
    return (last_hidden[0] * mask[:, None]).sum(0) / mask.sum()  # mean pool → [384]

def cosine(a, b):
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

e1 = embed("had dinner at Sharkey's with friends last night")
e2 = embed("went out for food and drinks with the group")
e3 = embed("quarterly earnings report and budget forecast")

print(f"dinner <-> dinner : {cosine(e1, e1):.4f}  (expect 1.0000)")
print(f"dinner <-> food   : {cosine(e1, e2):.4f}  (expect high  >0.7)")
print(f"dinner <-> finance: {cosine(e1, e3):.4f}  (expect low   <0.4)")

assert cosine(e1, e2) > cosine(e1, e3), "Similarity ordering wrong — model may not have converted correctly"
print("\nSanity check passed!")

In [ ]:
# Cell 7 — Download
# Drop the .tflite into core-logic/src/main/assets/ replacing the float16 file.
# vocab.txt does NOT need replacing — same BERT WordPiece tokenizer.
try:
    from google.colab import files
    files.download(TFLITE_PATH)
except ImportError:
    # Kaggle: file appears in the Output tab
    import shutil
    shutil.copy(TFLITE_PATH, "/kaggle/working/snowflake-arctic-embed-xs.tflite")
    print("Kaggle: grab the file from the Output tab on the right.")